### This Notebook provide the code to analyse equivalent/non-equivalent mutants, to determine the ideal thresholds for trace distance and fidelity. 


In [ ]:
import pandas as pd

# Load CSV results

In [ ]:
mutants_file_path = "results_custom_brisbane/results_normal.csv"
equiv_file_path =  "results_custom_brisbane/results_equiv.csv"

column_names = ['Name', 'Input', 'Ideal_chisquare', 'Ideal_hellinger', 'Ideal_jensenshannon', 'Ideal_trace', 'Ideal_fidelity',
                'Ideal_expectation', 'Killed_IC', 'Killed_IH', 'Killed_IJ', 'Killed_IT', 'Killed_IF', 'Killed_IE']
    
mutants_df = pd.read_csv(mutants_file_path, names=column_names, header=0)
equiv_df = pd.read_csv(equiv_file_path, names=column_names, header=0)

# Display the DataFrame to verify
print(mutants_df.head())
print(equiv_df.head())

# Print number of mutants

In [ ]:
unique_count = mutants_df['Name'].nunique()
print("Number of executions in the mutant set: ", len(mutants_df))
print("Number of mutants in the mutant set: ", unique_count)


unique_count = equiv_df['Name'].nunique()
print("Number of executions in the equivalent set: ", len(equiv_df))
print("Number of mutants in the equivalent set: ", unique_count)


# Check for Equivalence 

In [ ]:
trace_error_rounded = 1E-13
fidelity_error_rounded = 1E-14

In [ ]:
def classify_mutants(mutants_df):
    
    # Find killed mutants based on Ideal_expectation
    killed = mutants_df[mutants_df['Ideal_expectation'] != 0]
    killed = killed['Name'].unique()

    # Find mixed mutants based on Ideal_expectation and Ideal_fidelity
    df_mutant_candidates = mutants_df.groupby('Name').filter(lambda x: (x['Ideal_expectation'] == 0).all())
    df_filtered = df_mutant_candidates[(1 - df_mutant_candidates['Ideal_fidelity'] > fidelity_error_rounded)]
    mixed = df_filtered['Name'].unique()

    # Find survived mutants that are neither killed nor mixed
    survived = mutants_df[~mutants_df['Name'].isin(killed) & ~mutants_df['Name'].isin(mixed)]
    survived = survived['Name'].unique()
    
    return killed, survived, mixed


In [ ]:
def check_sizes(mutants_df, killed, survived, mixed):
    print("Total mutants: ", len(mutants_df['Name'].unique()))
    print("Killed mutants: ", len(killed))
    print("Equivalent mutants: ", len(survived))
    print("Mixed mutants: ", len(mixed))
    print("Check total: ", len(mixed) + len(killed) + len(survived))

In [ ]:
killed, survived, mixed = classify_mutants(equiv_df)
print("Mutants in equivalent set:")
total = len(equiv_df['Name'].unique())
print("Total mutants", total)
#print(killed)
#print(survived)
#print(mixed)
#check_sizes(equiv_df, killed, survived, mixed)
killed_or_mixed = set(killed).union(set(mixed))
#print(killed_or_mixed)
print("Killed mutants", len(killed_or_mixed))
percentage = round(len(killed_or_mixed) / total * 100, 2)
print("Percentage to move: " + str(percentage) + "%")

print("")

killed, survived, mixed = classify_mutants(mutants_df)
print("Mutants in normal set:")
total = len(mutants_df['Name'].unique())
print("Total mutants", total)
#print("Killed by expectation", killed)
#print(survived)
print("Survived mutants", len(survived))
#print(mixed)

killed_or_mixed = set(killed).union(set(mixed))

percentage = round(len(survived) / total * 100, 2)
print("Percentage to move: " + str(percentage) + "%")

#print(killed_or_mixed)

#check_sizes(mutants_df, killed, survived, mixed)

In [ ]:
def further_analysis(mutants_df):
    # Shows that expectation > fidelity and trace
    df_filtered = mutants_df[(mutants_df['Ideal_expectation'] != 0)]
    nb_non_equivalent_executions = len(df_filtered)
    
    fidelity_mutants = df_filtered[(1 - df_filtered['Ideal_fidelity'] > fidelity_error_rounded)]
    fidelity_equivalence = len(fidelity_mutants) == nb_non_equivalent_executions
    
    trace_mutants = df_filtered[(df_filtered['Ideal_trace'] > trace_error_rounded)]
    trace_equivalence = len(trace_mutants) == nb_non_equivalent_executions
    
    # Find rows in expectation_mutants that are not in trace_mutants
    # only_in_expectation_mutants = (df_filtered)[~df_filtered.index.isin(trace_mutants.index)]
    # print(only_in_expectation_mutants)
    
    print("Does fidelity detect the mutation when expectation does?", fidelity_equivalence)
    print("Does trace distance detect the mutation when expectation does?", trace_equivalence)
    
    df_filtered_names = set(df_filtered['Name'].unique())
    trace_mutants_names = set(trace_mutants['Name'].unique())
    only_in_df_filtered = df_filtered_names - trace_mutants_names
    only_in_trace_mutants = trace_mutants_names - df_filtered_names
    both_sets_empty = not only_in_df_filtered and not only_in_trace_mutants
    print("Are all individual mutants detected by both trace distance and expectation?", both_sets_empty)
    
    # Shows that fidelity > trace and  expectation != fidelity
    df_filtered = mutants_df[(mutants_df['Ideal_expectation'] == 0) & (1 - mutants_df['Ideal_fidelity'] > fidelity_error_rounded)]
    df_fidelity_ideal = df_filtered[(df_filtered['Ideal_trace'] > trace_error_rounded)]
    df_ideal = mutants_df[(mutants_df['Ideal_expectation'] == 0) & (mutants_df['Ideal_trace'] > trace_error_rounded)]
    
    trace_equivalence = len(df_fidelity_ideal) == len(df_ideal)
    
    print("Are mutants discover by trace distance also detected by fidelity?", trace_equivalence)

In [ ]:
print("Equivalent set analysis:")
further_analysis(equiv_df)
print("\nNormal set analysis:")
further_analysis(mutants_df)